# Galactic-binary parameter inference with `jexplore`

Bayesian inference of the parameters of a **single galactic binary (GB)** from a
one-month LISA A/E/T frequency-domain datastream, using the affine-invariant
ensemble sampler [`jexplore`](https://pypi.org/project/jexplore/).

The datastream (clean signal + instrumental noise) and the noise PSD are produced
**entirely by the helpers in [`src/lisa.py`](src/lisa.py)** — `clean_signal`,
`sample_noise` and `noise_psd` — the same code that feeds the diffusion model in
[train.py](train.py). This notebook is the MCMC cross-check of that pipeline.

We sample the four parameters **`θ = [f₀, ḟ, A, ψ]`**, holding the sky position and
orientation `(ra, dec, ι, φ₀)` fixed at their injected values, and recover **all
four constrained**.

Inspired by [glitch_and_gb.ipynb](glitch_and_gb.ipynb) but with the glitch sector
removed — here there is *only* a galactic binary.

## Choosing a constrainable injection

`ḟ` is only measurable if the frequency chirp builds up enough phase over the
observation: its precision scales as `σ_ḟ ∝ 1 / (SNR · T_obs²)`. With the
`lisa.prior_inverse_cdf` training scale (`ḟ ≤ 4×10⁻¹⁸`) over a short baseline the
chirp is unresolvable and the `ḟ` marginal collapses onto the prior.

So, exactly as `glitch_and_gb.ipynb` does (it uses a *heavy-chirp* `ḟ = 10⁻¹⁷` over a
**1-year** baseline and **raises the `ḟ`/`A` prior caps** above the training prior),
we pick a heavy chirp here too. We run at **1-month** resolution (this GPU only has
~2 GB free; the year-long grid OOMs), and since `σ_ḟ ∝ 1/T_obs²` a month needs a
correspondingly heavier chirp `ḟ = 10⁻¹⁶` to reach the same resolvability — then all
four parameters are data-constrained. `f₀`, `A`, `ψ` are constrained at the
`lisa.prior_inverse_cdf` scale already.

## Likelihood normalisation

`lisa.sample_noise` colours the frequency-domain noise to the **physical one-sided
PSD** `S(f) = lisa.noise_psd(channel)`: each rfft bin has
`E[|n_f|²] = (N / 2·dt)·S(f)` with `N = T_obs/dt`, so the estimator
`(2·dt/N)·|rfft(n)|²` is an unbiased `S(f)`. The matching Gaussian log-likelihood
therefore carries a `(2·dt/N)` factor:

`log L(θ) = −(2·dt/N) · Σ_f Σ_ch |d_f − h_f(θ)|² / S(f)`

(At the truth this gives `log L ≈ −(#bins × #channels)`, the expected χ² with 2 dof
per complex bin, which we check below.) The physical optimal SNR of the injection is
`lisa.optimal_snr(params, T_obs, dt)` (`SNR² = 4·Δf·Σ dt²·|h̃|²/S`), **not** the raw
`√(Σ|h|²/S)` used in the SNR cells below — the latter omits the same `~N/2dt` factor
and overstates the SNR by ~20×.


In [ ]:
import sys, os, time
sys.path.insert(0, os.path.abspath(""))   # so `import src.lisa` works from the notebook
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")  # allocate on demand (shared GPU)

import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import jax.random as jr
import numpy as np
import corner
import matplotlib.pyplot as plt
import matplotlib.lines as mlines

from src import lisa, noise_utils

from jexplore.sampler import JaxSampler, Steps
from jexplore.sampling import EpochMH, SamplingMH
from jexplore.steps import Stretch
from jexplore.backends import DefaultBackend

print("JAX backend:", jax.default_backend(), "| devices:", jax.devices())

## 1. Observation, true parameters and datastream

Everything below comes from `src/lisa.py`. The clean A/E/T signal is
`lisa.clean_signal` and the instrumental noise realisation is `lisa.sample_noise`;
both return the rFFT on the same cropped frequency grid, so `data = signal + noise`.

In [ ]:
# ---- observation ----
T_OBS = lisa.YEAR_s            # observation time (matches train.py)
DT    = lisa.SAMPLING_STEP_s    # Nyquist step for the 3 mHz band (~167 s)
N_SLOW = 256                    # points for the slow TDI response (clean_signal `n`)
NCROP  = 32                     # frequency-grid crop
SEED   = 0

# ---- TEST: a single sample drawn from a training batch ----
# Sanity check that the generative pipeline (clean_signal + physical sample_noise)
# produces a sensible optimal SNR for a random prior draw.
_, _, _, _, _, _, params, datastream = lisa.get_train_batch(jr.PRNGKey(SEED),
    batch_size = 10,
    n_sources = 1,
    t_obs = T_OBS,
    dt = DT,
    noise_scale = 1.0,
)

n_samples = int(T_OBS / DT)
freq = jnp.fft.rfftfreq(n_samples, DT)
data = jnp.fft.rfft(datastream[0], axis=-2)      # the single batch sample (signal+noise)

# physical matched-filter SNR of this batch sample (lisa.optimal_snr, not the raw sum)
snr = float(lisa.optimal_snr(params[0], T_OBS, DT))
print(f"data shape {data.shape},  {freq.shape[0]} freq bins")
print(f"single batch-sample optimal SNR = {snr:.4f}")


In [ ]:
# ---- INJECTION under test: the sampled parameters of arXiv:2606.20269 ----
# The four sampled parameters (f0, fdot, A, phi0) are set to the paper's injection,
# read from lisa_gb_demo.npz `truth_display` = [log10 f0, fdot, log10 A, phi0]:
#     log10 f0 = -2.85824130,  fdot = 9.38982036e-15,
#     log10 A  = -23.2713149,  phi0 = 1.74379807
# The sky/orientation angles (ra, dec, psi, iota) are NOT stored in the npz nor given
# in the paper abstract, so they are kept from the earlier example values.
# T_obs / dt are taken from the npz so this reproduces the paper setup.
T_OBS_new = 21845333.333333332      # = lisa_gb_demo.npz t_obs (pairs with params_new)
T_OBS     = T_OBS_new               # inference uses this T_obs from here on
n_samples = int(T_OBS / DT)
freq = jnp.fft.rfftfreq(n_samples, DT)

# [f0, fdot, A, ra, dec, psi, iota, phi0]
params_new = jnp.array([10 ** -2.85824130,  9.38982036e-15,  10 ** -23.2713149,
                        5.38175072e+00, -4.38943044e-01, 3.02757704e+00,
                        2.27219990e+00,  1.74379807e+00])

# make params_new the injected truth for the sampler below
F0_TRUE, FDOT_TRUE, A_TRUE = float(params_new[0]), float(params_new[1]), float(params_new[2])
RA_TRUE, DEC_TRUE          = float(params_new[3]), float(params_new[4])
PSI_TRUE                   = float(params_new[5])   # fixed (not inferred)
IOTA_TRUE, PHI0_TRUE       = float(params_new[6]), float(params_new[7])

# datastream = clean signal + physical instrumental noise (both from lisa)
signal = jnp.fft.rfft(lisa.clean_signal(params_new[None], T_OBS, DT), axis=-2)
noise  = jnp.fft.rfft(lisa.sample_noise(jr.PRNGKey(SEED), T_OBS, DT), axis=-2)
data   = signal + noise

f_safe = jnp.where(freq > 0, freq, 1.0)                       # avoid f=0 in the PSD
psd  = jnp.stack([lisa.noise_psd(c)(f_safe) for c in "AET"], axis=-1)   # (F, 3)
mask = (freq > 0)[:, None]                                    # drop the DC bin

snr = float(lisa.optimal_snr(params_new[None], T_OBS, DT))
print(f"data shape {data.shape},  {freq.shape[0]} freq bins")
print(f"params_new optimal SNR = {snr:.4f}   (freq-domain; paper WDM value ~20.7)")


## 2. Parameterisation, prior and likelihood

We sample the same four parameters as **arXiv:2606.20269**:
`θ = [log f₀, log ḟ, log A, φ₀]` — natural-log for the three log-uniform GB
amplitudes/frequencies and linear for the initial phase `φ₀`. Sky position
`(ra, dec)`, polarisation `ψ` and inclination `ι` are held fixed at the injected
values. The prior is flat in `θ`; the `ḟ` and `A` caps are raised above the
`lisa.prior_inverse_cdf` training bounds to bracket the injection, and `φ₀` is
uniform over `(-π, π)`.

In [ ]:
# Prior bounds — match arXiv:2606.20269 sampled set [f0, fdot, A, phi0].
# f0 as in lisa.prior_inverse_cdf; fdot/A caps raised to bracket the injection
# (fdot = 9.39e-15 exceeds the training cap); phi0 uniform over (-pi, pi).
F0_MIN,   F0_MAX   = 1e-4,  3e-3
FDOT_MIN, FDOT_MAX = 1e-18, 1e-13
A_MIN,    A_MAX    = 1e-25, 1.7e-22
PHI0_MIN, PHI0_MAX = -float(jnp.pi), float(jnp.pi)

DIM    = 4
labels = ["log f0", "log fdot", "log A", "phi0"]

theta_true = jnp.array([jnp.log(F0_TRUE), jnp.log(FDOT_TRUE), jnp.log(A_TRUE), PHI0_TRUE])


def to_params(theta):
    """Sampling vector θ = [log f0, log fdot, log A, phi0] -> full (1, 8) GB params,
    with sky position, polarisation psi and inclination held at the injected values
    (matches the arXiv:2606.20269 corner, which samples phi0 rather than psi)."""
    f0, fdot, A = jnp.exp(theta[0]), jnp.exp(theta[1]), jnp.exp(theta[2])
    phi0 = theta[3]
    return jnp.array([[f0, fdot, A, RA_TRUE, DEC_TRUE, PSI_TRUE, IOTA_TRUE, phi0]])


# Normalised Gaussian log-likelihood. `lisa.sample_noise` now colours the noise to the
# physical one-sided PSD, so each rfft bin has variance (N/2dt)·S(f); the matching
# exponent carries a (2·dt/N) factor (N = time-series length = 2·(#rfft bins − 1)).
# Without it the noise weight is ~N/2dt ≈ 400× off and the χ²/SNR are inflated.
# (`h` is rfft'd here to match how `data` is built in the cells above.)
N_TIME      = 2 * (freq.shape[0] - 1)
LOGLIK_NORM = 2.0 * DT / N_TIME


@jax.jit
def log_lik(theta):
    h = jnp.fft.rfft(lisa.clean_signal(to_params(theta), T_OBS, DT), axis=-2)
    r = data - h
    return -LOGLIK_NORM * jnp.sum(jnp.where(mask, jnp.abs(r) ** 2 / psd, 0.0))


@jax.jit
def log_prior(theta):
    ok = (
        (theta[0] >= jnp.log(F0_MIN))   & (theta[0] <= jnp.log(F0_MAX))
      & (theta[1] >= jnp.log(FDOT_MIN)) & (theta[1] <= jnp.log(FDOT_MAX))
      & (theta[2] >= jnp.log(A_MIN))    & (theta[2] <= jnp.log(A_MAX))
      & (theta[3] >= PHI0_MIN)          & (theta[3] <= PHI0_MAX)
    )
    return jnp.where(ok, 0.0, -jnp.inf)


# Sanity: at the truth log L ≈ -(#bins × #channels) (χ², 2 dof per complex bin).
# The (2·dt/N) normalisation is exactly what makes this hold with the physical noise.
n_terms = int(jnp.sum(jnp.broadcast_to(mask, data.shape)))
print(f"log L(truth)      = {float(log_lik(theta_true)):.1f}")
print(f"-(bins×channels)  = {-n_terms}")
print(f"log L(truth+δ)    = {float(log_lik(theta_true + jnp.array([1e-3, 0., 0.1, 0.2]))):.1f}  (must be lower)")


## 3. Ensemble sampling with `jexplore`

A gradient-free affine-invariant `Stretch` ensemble. The `lisa.clean_signal` model
places the GB on an integer frequency bin (via `get_kmin`), so the likelihood is not
smoothly differentiable in `f₀` — a gradient-free ensemble move is the right choice.
Walkers are initialised in a tight ball around the (known) injection.

In [ ]:
N_WALKERS = 16
N_BURN    = 300
N_SAMP    = 1_000

# initialisation scatter per parameter (f0 is extremely well constrained → tiny)
sigma0 = jnp.array([1e-6, 0.1, 0.02, 0.02])
p0 = theta_true + sigma0 * jr.normal(jr.key(SEED + 1), (N_WALKERS, DIM))

sampling = SamplingMH(
    nwalker=N_WALKERS, temps=jnp.array([1.0]),
    loglik=log_lik, logprior=log_prior, dim=DIM,
)
steps   = Steps([{Stretch(permute=True).builder: 1.0}])
iepoch  = EpochMH({"p": p0})
backend = DefaultBackend(burn=N_BURN, inmem_epochs=1)

print(f"Running jexplore  ({N_WALKERS} walkers × {N_BURN + N_SAMP} iters)...")
t0 = time.time()
JaxSampler(sampling, steps, backend).run(iepoch, niters=N_BURN + N_SAMP, nepoch=1, seed=SEED + 1)
# backend stores (N_WALKERS, DIM, N_SAMP) -> flatten to (N_WALKERS*N_SAMP, DIM)
raw = backend.get_samples()["p"]
chain = np.asarray(raw.transpose(0, 2, 1).reshape(-1, DIM))
print(f"  done in {time.time() - t0:.1f} s,  {chain.shape[0]:,} samples")

In [ ]:
# Posterior summary in physical units
print(f"{'parameter':12s}  {'median':>14s}  {'truth':>14s}")
print("-" * 44)
meds = np.median(chain, axis=0)
phys = lambda v: (np.exp(v[0]), np.exp(v[1]), np.exp(v[2]), v[3])
names = ["f0 (Hz)", "fdot (Hz/s)", "A", "phi0 (rad)"]
for n, m, t in zip(names, phys(meds), phys(np.asarray(theta_true))):
    print(f"{n:12s}  {m:14.4e}  {t:14.4e}")


## 4. Corner plot

Marginal posteriors in the **display units of arXiv:2606.20269**:
`f₀ − f₀ⁱⁿʲ` in nHz (panel centred at 0), `ḟ` in units of `10⁻¹⁵ Hz/s` (linear, not
log), `log₁₀ A`, and `φ₀` in radians. The injected truth is the black line; the
title reports the frequency-domain optimal SNR.

In [ ]:
# Corner plot in the display units of arXiv:2606.20269:
#   f0   -> (f0 - f0_inj) in nHz  (panel centred at 0)
#   fdot -> units of 1e-15 Hz/s   (linear, not log)
#   A    -> log10 A
#   phi0 -> rad
# chain columns are θ = [ln f0, ln fdot, ln A, phi0] (natural logs from theta_true).
to_plot = lambda a: np.column_stack([
    (np.exp(a[:, 0]) - F0_TRUE) * 1e9,   # f0 - f0_inj  [nHz]
    np.exp(a[:, 1]) * 1e15,              # fdot         [1e-15 Hz/s]
    np.log10(np.exp(a[:, 2])),           # log10 A
    a[:, 3],                             # phi0         [rad]
])
plot_samples = to_plot(chain)
plot_truth   = [0.0, FDOT_TRUE * 1e15, np.log10(A_TRUE), float(PHI0_TRUE)]
plot_labels  = [r"$f_0 - f_0^{\mathrm{inj}}$ [nHz]",
                r"$\dot f\ [10^{-15}\,\mathrm{Hz/s}]$",
                r"$\log_{10} A$",
                r"$\phi_0$ [rad]"]

plt.close("all")
fig = corner.corner(
    plot_samples,
    labels=plot_labels,
    truths=plot_truth,
    truth_color="black",
    color="C0",
    plot_datapoints=False,
    smooth=1.0,
    bins=30,
    levels=(0.5, 0.9),
    max_n_ticks=3,
    show_titles=True,
    title_fmt=".3f",
    hist_kwargs={"density": True},
    label_kwargs={"fontsize": 14},
)
fig.legend(
    handles=[
        mlines.Line2D([], [], color="C0", label=f"jexplore (SNR={snr:.1f})"),
        mlines.Line2D([], [], color="black", label="injected truth"),
    ],
    loc="upper right", fontsize=13, frameon=False,
)
fig.suptitle("Galactic-binary posterior (arXiv:2606.20269 units)", y=1.02)
#fig.savefig("GB_inference_corner.pdf", bbox_inches="tight")
plt.show()
#print("saved GB_inference_corner.pdf")
